In [4]:
from pathlib import Path
import pandas as pd

# =========================
# 🔧 你只需要改这里
# =========================
CSV_FILES = [
    "/home/clara/Research/fyp_fl_smart_home/results/uci_fedavg_r50_20260114_215845.csv",
    "/home/clara/Research/fyp_fl_smart_home/results/uci_fedavg_r50_20260114_221036.csv",   # rpi5
    "/home/clara/Research/fyp_fl_smart_home/results/uci_fedavg_r50_20260114_221933.csv",   # jetson_nano
    "/home/clara/Research/fyp_fl_smart_home/results/uci_fedavg_r50_20260114_222555.csv",   # coral
    "/home/clara/Research/fyp_fl_smart_home/results/uci_fedavg_r50_20260114_223300.csv",   # esp32
]

LAST_N = 5   # 用最后 N 轮取平均（论文里建议 5 或 10）
# =========================


rows = []

for file in CSV_FILES:
    path = Path(file)
    if not path.exists():
        print(f"[WARN] File not found: {path}")
        continue

    df = pd.read_csv(path)

    # 基本安全检查
    required = {"round", "fit_sim_s", "energy_wh", "cpu_pct", "device_key"}
    if not required.issubset(df.columns):
        print(f"[SKIP] {path.name} missing edge columns")
        continue

    max_r = int(df["round"].max())
    tail = df[df["round"] >= max_r - (LAST_N - 1)]

    device_key = tail["device_key"].dropna().iloc[0]
    device_profile = (
        tail["device_profile"].dropna().iloc[0]
        if "device_profile" in tail.columns and tail["device_profile"].notna().any()
        else ""
    )

    rows.append({
        "device_key": device_key,
        "device_profile": device_profile,
        "latency_fit_sim_s": float(tail["fit_sim_s"].mean()),
        "energy_wh": float(tail["energy_wh"].mean()),
        "cpu_pct": float(tail["cpu_pct"].mean()),
    })

summary_df = pd.DataFrame(rows)

if summary_df.empty:
    raise ValueError("No valid CSV loaded. Check file paths and columns.")

summary_df


,device_key,device_profile,latency_fit_sim_s,energy_wh,cpu_pct
0,rpi4,Raspberry Pi 4,9.347859,0.018176,103.290318
1,rpi5,Raspberry Pi 5,6.714988,0.022383,102.679159
2,jetson_nano,NVIDIA Jetson Nano,4.448032,0.012356,102.568925
3,coral,Google Coral,5.206814,0.005785,102.823120
4,esp32,ESP32,186.152033,0.025854,102.834546


In [14]:
from pathlib import Path
import os
import pandas as pd

# =========================
# 你只需要改这里
# =========================
CSV_FILES = [
    "/home/clara/Research/fyp_fl_smart_home/results/uci_fedavg_r50_20260114_215845.csv",  # rpi4
    "/home/clara/Research/fyp_fl_smart_home/results/uci_fedavg_r50_20260114_221036.csv",  # rpi5
    "/home/clara/Research/fyp_fl_smart_home/results/uci_fedavg_r50_20260114_221933.csv",  # jetson_nano
    "/home/clara/Research/fyp_fl_smart_home/results/uci_fedavg_r50_20260114_222555.csv",  # coral
    "/home/clara/Research/fyp_fl_smart_home/results/uci_fedavg_r50_20260114_223300.csv",  # esp32
]
LAST_N = 5
# =========================

# 自动定位项目根目录
ROOT = Path(os.getcwd()).resolve()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent
RESULTS_DIR = ROOT / "results"

rows = []

for name in CSV_FILES:
    path = RESULTS_DIR / name
    if not path.exists():
        print(f"[WARN] File not found: {path}")
        continue

    df = pd.read_csv(path)

    required = {
        "round", "accuracy", "f1",
        "fit_sim_s", "energy_wh", "cpu_pct",
        "device_key"
    }
    if not required.issubset(df.columns):
        print(f"[SKIP] {path.name} missing columns: {required - set(df.columns)}")
        continue

    max_r = int(df["round"].max())
    tail = df[df["round"] >= max_r - (LAST_N - 1)]

    rows.append({
        "device_key": tail["device_key"].dropna().iloc[0],
        "device_profile": tail["device_profile"].dropna().iloc[0],
        "accuracy": float(tail["accuracy"].mean()),
        "f1_score": float(tail["f1"].mean()),
        "latency_fit_sim_s": float(tail["fit_sim_s"].mean()),
        "energy_wh": float(tail["energy_wh"].mean()),
        "cpu_pct": float(tail["cpu_pct"].mean()),
    })

summary_df = pd.DataFrame(rows)

final_table = (
    summary_df
    .groupby(["device_key", "device_profile"], as_index=False)
    .mean(numeric_only=True)
    #.sort_values("latency_fit_sim_s")
    .round({
        "accuracy": 3,
        "f1_score": 3,
        "latency_fit_sim_s": 3,
        "energy_wh": 6,
        "cpu_pct": 2,
    })
)

final_table


,device_key,device_profile,accuracy,f1_score,latency_fit_sim_s,energy_wh,cpu_pct
0,coral,Google Coral,0.834,0.823,5.207,0.005785,102.82
1,esp32,ESP32,0.795,0.763,186.152,0.025854,102.83
2,jetson_nano,NVIDIA Jetson Nano,0.783,0.747,4.448,0.012356,102.57
3,rpi4,Raspberry Pi 4,0.837,0.820,9.348,0.018176,103.29
4,rpi5,Raspberry Pi 5,0.837,0.829,6.715,0.022383,102.68


In [15]:
from pathlib import Path
import os
import pandas as pd

# =========================
# 你只需要改这里
# =========================
CSV_FILES = [
    "/home/clara/Research/fyp_fl_smart_home/results/uci_fedprox_r50_20260114_230225.csv",  # rpi4
    "/home/clara/Research/fyp_fl_smart_home/results/uci_fedprox_r50_20260114_231201.csv",  # rpi5
    "/home/clara/Research/fyp_fl_smart_home/results/uci_fedprox_r50_20260114_231856.csv",  # jetson_nano
    "/home/clara/Research/fyp_fl_smart_home/results/uci_fedprox_r50_20260114_232431.csv",  # coral
    "/home/clara/Research/fyp_fl_smart_home/results/uci_fedprox_r50_20260114_234340.csv",  # esp32
]
LAST_N = 5
# =========================

# 自动定位项目根目录
ROOT = Path(os.getcwd()).resolve()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent
RESULTS_DIR = ROOT / "results"

rows = []

for name in CSV_FILES:
    path = RESULTS_DIR / name
    if not path.exists():
        print(f"[WARN] File not found: {path}")
        continue

    df = pd.read_csv(path)

    required = {
        "round", "accuracy", "f1",
        "fit_sim_s", "energy_wh", "cpu_pct",
        "device_key"
    }
    if not required.issubset(df.columns):
        print(f"[SKIP] {path.name} missing columns: {required - set(df.columns)}")
        continue

    max_r = int(df["round"].max())
    tail = df[df["round"] >= max_r - (LAST_N - 1)]

    rows.append({
        "device_key": tail["device_key"].dropna().iloc[0],
        "device_profile": tail["device_profile"].dropna().iloc[0],
        "accuracy": float(tail["accuracy"].mean()),
        "f1_score": float(tail["f1"].mean()),
        "latency_fit_sim_s": float(tail["fit_sim_s"].mean()),
        "energy_wh": float(tail["energy_wh"].mean()),
        "cpu_pct": float(tail["cpu_pct"].mean()),
    })

summary_df = pd.DataFrame(rows)

final_table = (
    summary_df
    .groupby(["device_key", "device_profile"], as_index=False)
    .mean(numeric_only=True)
    #.sort_values("latency_fit_sim_s")
    .round({
        "accuracy": 3,
        "f1_score": 3,
        "latency_fit_sim_s": 3,
        "energy_wh": 6,
        "cpu_pct": 2,
    })
)

final_table


,device_key,device_profile,accuracy,f1_score,latency_fit_sim_s,energy_wh,cpu_pct
0,coral,Google Coral,0.753,0.735,5.382,0.005980,103.29
1,esp32,ESP32,0.737,0.706,234.935,0.032630,104.55
2,jetson_nano,NVIDIA Jetson Nano,0.692,0.642,4.582,0.012728,102.40
3,rpi4,Raspberry Pi 4,0.698,0.650,9.656,0.018776,102.94
4,rpi5,Raspberry Pi 5,0.728,0.704,6.928,0.023093,102.80


In [16]:
from pathlib import Path
import os
import pandas as pd

# =========================
# 你只需要改这里
# =========================
CSV_FILES = [
    "/home/clara/Research/fyp_fl_smart_home/results/uci_feddc_r50_20260114_235148.csv",  # rpi4
    "/home/clara/Research/fyp_fl_smart_home/results/uci_feddc_r50_20260115_000404.csv",  # rpi5
    "/home/clara/Research/fyp_fl_smart_home/results/uci_feddc_r50_20260115_002016.csv",  # jetson_nano
    "/home/clara/Research/fyp_fl_smart_home/results/uci_feddc_r50_20260115_002559.csv",  # coral
    "/home/clara/Research/fyp_fl_smart_home/results/uci_feddc_r50_20260115_003142.csv",  # esp32
]
LAST_N = 5
# =========================

# 自动定位项目根目录
ROOT = Path(os.getcwd()).resolve()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent
RESULTS_DIR = ROOT / "results"

rows = []

for name in CSV_FILES:
    path = RESULTS_DIR / name
    if not path.exists():
        print(f"[WARN] File not found: {path}")
        continue

    df = pd.read_csv(path)

    required = {
        "round", "accuracy", "f1",
        "fit_sim_s", "energy_wh", "cpu_pct",
        "device_key"
    }
    if not required.issubset(df.columns):
        print(f"[SKIP] {path.name} missing columns: {required - set(df.columns)}")
        continue

    max_r = int(df["round"].max())
    tail = df[df["round"] >= max_r - (LAST_N - 1)]

    rows.append({
        "device_key": tail["device_key"].dropna().iloc[0],
        "device_profile": tail["device_profile"].dropna().iloc[0],
        "accuracy": float(tail["accuracy"].mean()),
        "f1_score": float(tail["f1"].mean()),
        "latency_fit_sim_s": float(tail["fit_sim_s"].mean()),
        "energy_wh": float(tail["energy_wh"].mean()),
        "cpu_pct": float(tail["cpu_pct"].mean()),
    })

summary_df = pd.DataFrame(rows)

final_table = (
    summary_df
    .groupby(["device_key", "device_profile"], as_index=False)
    .mean(numeric_only=True)
    #.sort_values("latency_fit_sim_s")
    .round({
        "accuracy": 3,
        "f1_score": 3,
        "latency_fit_sim_s": 3,
        "energy_wh": 6,
        "cpu_pct": 2,
    })
)

final_table


,device_key,device_profile,accuracy,f1_score,latency_fit_sim_s,energy_wh,cpu_pct
0,coral,Google Coral,0.385,0.274,5.257,0.005841,103.33
1,esp32,ESP32,0.375,0.263,186.844,0.025951,103.33
2,jetson_nano,NVIDIA Jetson Nano,0.379,0.266,21.810,0.060583,105.75
3,rpi4,Raspberry Pi 4,0.370,0.244,11.290,0.021953,104.00
4,rpi5,Raspberry Pi 5,0.376,0.262,38.852,0.129506,106.28


In [18]:
from pathlib import Path
import os
import pandas as pd

# =========================
# 你只需要改这里
# =========================
CSV_FILES = [
    "/home/clara/Research/fyp_fl_smart_home/results/uci_scaffold_r50_20260115_003743.csv",  # rpi4
    "/home/clara/Research/fyp_fl_smart_home/results/uci_scaffold_r50_20260115_004321.csv",  # rpi5
    "/home/clara/Research/fyp_fl_smart_home/results/uci_scaffold_r50_20260115_004910.csv",  # jetson_nano
    "/home/clara/Research/fyp_fl_smart_home/results/uci_scaffold_r50_20260115_005541.csv",  # coral
    "/home/clara/Research/fyp_fl_smart_home/results/uci_scaffold_r50_20260115_010131.csv",  # esp32
]
LAST_N = 5
# =========================

# 自动定位项目根目录
ROOT = Path(os.getcwd()).resolve()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent
RESULTS_DIR = ROOT / "results"

rows = []

for name in CSV_FILES:
    path = RESULTS_DIR / name
    if not path.exists():
        print(f"[WARN] File not found: {path}")
        continue

    df = pd.read_csv(path)

    required = {
        "round", "accuracy", "f1",
        "fit_sim_s", "energy_wh", "cpu_pct",
        "device_key"
    }
    if not required.issubset(df.columns):
        print(f"[SKIP] {path.name} missing columns: {required - set(df.columns)}")
        continue

    max_r = int(df["round"].max())
    tail = df[df["round"] >= max_r - (LAST_N - 1)]

    rows.append({
        "device_key": tail["device_key"].dropna().iloc[0],
        "device_profile": tail["device_profile"].dropna().iloc[0],
        "accuracy": float(tail["accuracy"].mean()),
        "f1_score": float(tail["f1"].mean()),
        "latency_fit_sim_s": float(tail["fit_sim_s"].mean()),
        "energy_wh": float(tail["energy_wh"].mean()),
        "cpu_pct": float(tail["cpu_pct"].mean()),
    })

summary_df = pd.DataFrame(rows)

final_table = (
    summary_df
    .groupby(["device_key", "device_profile"], as_index=False)
    .mean(numeric_only=True)
    #.sort_values("latency_fit_sim_s")
    .round({
        "accuracy": 3,
        "f1_score": 3,
        "latency_fit_sim_s": 3,
        "energy_wh": 6,
        "cpu_pct": 2,
    })
)

final_table


,device_key,device_profile,accuracy,f1_score,latency_fit_sim_s,energy_wh,cpu_pct
0,coral,Google Coral,0.355,0.215,5.200,0.005778,103.56
1,esp32,ESP32,0.371,0.249,184.491,0.025624,102.99
2,jetson_nano,NVIDIA Jetson Nano,0.384,0.272,4.399,0.012221,103.16
3,rpi4,Raspberry Pi 4,0.373,0.246,9.239,0.017964,103.03
4,rpi5,Raspberry Pi 5,0.384,0.272,6.654,0.022180,103.59
